# Module 7 — Unity Catalog Basics (Governance)
Exam domain: **Security, Governance, Monitoring & Testing**

Unity Catalog (three-level namespace, `GRANT`, external locations, lineage) is a
**Databricks workspace feature** tied to a metastore — it cannot run in Colab or
plain OSS Spark at all. This notebook explains the concepts and simulates the
`catalog.schema.table` naming convention locally with plain folders, so you can
practice the mental model. The Databricks version runs the real commands.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (SparkSession.builder
    .appName("Module7-UC-Simulation")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## Three-level namespace: `catalog.schema.table`
Unity Catalog adds a **catalog** level above the classic `schema.table` (Hive
metastore only has two levels). Simulate the same hierarchy with paths.

In [ ]:
import os
catalog, schema_name, table = "main", "sales", "orders"
path = f"/content/lake/{catalog}/{schema_name}/{table}"
os.makedirs(path, exist_ok=True)

df = spark.createDataFrame([(1, 100.0), (2, 50.0)], ["order_id", "amount"])
df.write.format("delta").mode("overwrite").save(path)
print(f"Simulated table address: {catalog}.{schema_name}.{table} -> {path}")
spark.read.format("delta").load(path).show()

## Governance concepts (pseudo-SQL — requires a real Databricks + UC metastore)
```sql
-- Catalogs and schemas
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.sales;

-- Access control: grant at catalog, schema, or table level
GRANT USE CATALOG ON CATALOG main TO `analysts`;
GRANT SELECT ON TABLE main.sales.orders TO `analysts`;
GRANT MODIFY ON TABLE main.sales.orders TO `data_engineers`;

-- External locations point at cloud storage governed centrally
CREATE EXTERNAL LOCATION landing_zone
  URL 's3://my-bucket/landing/'
  WITH (STORAGE CREDENTIAL my_storage_credential);
```

## Key exam concepts
- **Metastore**: one per region/account, sits above catalogs; `hive_metastore`
  is the legacy, workspace-local, two-level fallback.
- **Managed vs. external tables**: managed tables' data lives in the metastore's
  default storage and is deleted with `DROP TABLE`; external tables point at a
  location you control and survive the `DROP`.
- **Lineage**: Unity Catalog tracks table- and column-level lineage
  automatically across notebooks, DLT pipelines, and jobs — no manual tagging.
- **Row/column-level security**: dynamic views or row filters/column masks
  defined with `ALTER TABLE ... SET ROW FILTER` / `SET MASK`.